<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/942_EAASv3_DataGen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Your v2 Architecture Is Already Excellent

You essentially built a **mini production environment for evaluating agents**, not just a benchmark.

Your environment includes:

### Scenario layer

Customer journeys being tested.

* `journey_scenarios.json`

### Operational ground truth

Realistic system state.

* `orders.json`
* `logistics_api.json`
* `customers.json`

Example:
Orders contain shipping state, delivery status, warehouse flags, etc.

Customers include churn risk and loyalty tier signals.

These are exactly the signals the orchestrator uses to determine issue types.

---

### Decision logic layer

The orchestrator rules.

`orchestrator_decision_rules.json`

Example rules:

* lost package → refund + apology
* delay → update ETA
* delay + churn risk → escalate
* repeat item not received → refund + escalation

This rule system maps issue classification to:

* resolution path
* specialist agents
* final outcome.

That deterministic mapping is **perfect for evaluation scoring**.

---

### Specialist agent layer

`specialist_agents.json`

Agents include:

* refund_agent
* shipping_update_agent
* apology_message_agent
* escalation_agent.

These simulate real production micro-services.

---

### Evaluation telemetry layer

You already have the **most important file** for v3:

`run_summary_metrics.json`

Example metrics:

* pass rate
* classification accuracy
* path accuracy
* outcome accuracy
* high risk failures
* human review rate
* latency metrics.

And you already have **6 runs of history**, which is ideal for MVP trend analysis.

---

# 2. What This Means: v3 Requires Very Little New Data

Because you already store **run-level history**, v3 can focus on:

**trend intelligence**, not dataset explosion.

Your current run history already shows:

| Date   | Pass Rate |
| ------ | --------- |
| Dec 20 | 0.75      |
| Dec 27 | 0.82      |
| Jan 03 | 0.90      |
| Jan 10 | 0.88      |
| Jan 17 | 0.80      |
| Jan 24 | 0.92      |

This alone enables:

* regression detection
* recovery analysis
* stability tracking
* release validation.

All from the existing file.

---

# 3. The Only Missing Layer: Scenario-Level History

Right now you have **run summary metrics**.

But to diagnose problems, the orchestrator needs:

```
which scenarios failed?
```

That is the **only critical missing dataset.**

---

# 4. The 3 New Files for v3 (Lean MVP)

These are the **only additions I recommend.**

---

# Dataset 1 — Scenario Results History

`scenario_results_history.json`

Structure:

```
{
 "run_id": "RUN_2026_01_24",
 "scenario_id": "SCN_004",

 "predicted_issue_type": "delivery_delay",
 "expected_issue_type": "delivery_delay",

 "issue_classification_correct": true,
 "resolution_path_correct": true,
 "outcome_correct": true,

 "latency_ms": 740,

 "human_review_required": false,

 "hallucination_flag": false,
 "policy_violation_flag": false
}
```

This enables:

* root cause analysis
* scenario-level trends
* failure clustering.

Example insight:

```
delivery_delay_with_churn_risk
accuracy trend ↓ across last 3 runs
```

Now the evaluator can explain **why runs changed**.

---

# Dataset 2 — Trigger History

`trigger_history.json`

Example:

```
{
 "run_id": "RUN_2026_01_17",

 "accuracy_trigger": false,
 "latency_trigger": false,
 "churn_handling_trigger": true,
 "policy_trigger": false,

 "total_triggers": 1,
 "persistent_triggers": 1
}
```

Now you can track:

* new triggers
* persistent triggers
* resolved triggers.

---

# Dataset 3 — Scenario Severity Catalog

`scenario_catalog_enriched.json`

Extend your scenario dataset with:

```
{
 "scenario_id": "SCN_005",
 "severity_weight": 5,
 "business_risk": "high",
 "refund_exposure_usd": 120,
 "customer_risk": "high",
 "failure_category": "lost_package"
}
```

Why this matters:

Not all failures are equal.

Failing a **lost package** is worse than failing a **status check**.

Severity weighting enables **risk scoring**.

---

# 5. What the v3 Agent Will Now Be Able to Do

With only these small additions, the orchestrator can produce powerful analysis.

---

## Performance Trend

```
PASS RATE TREND

Dec 20  → 0.75
Dec 27  → 0.82
Jan 03  → 0.90
Jan 10  → 0.88
Jan 17  → 0.80
Jan 24  → 0.92

Trajectory: improving with temporary churn-logic regression
```

---

## Risk Trajectory

```
HIGH-RISK FAILURES

Dec 20 → 3
Dec 27 → 2
Jan 03 → 1
Jan 10 → 2
Jan 17 → 3
Jan 24 → 0
```

Interpretation:

```
temporary regression detected
resolved in latest run
```

---

## Scenario Failure Concentration

```
TOP FAILURES

delivery_delay_with_churn_risk → 3 failures
repeat_item_not_received → 1 failure
warehouse_delay → 1 failure
```

---

## Regression Detection

Example insight:

```
Jan 17 run detected regression
root cause:

churn risk thresholds adjusted
delivery_delay_with_churn_risk classification dropped
```

---

# 6. Final v3 Dataset Structure (Clean and Minimal)

Current datasets:

```
customers.json
orders.json
logistics_api.json
marketing_signals.json

journey_scenarios.json

specialist_agents.json
orchestrator_decision_rules.json

evaluation_runs.json
run_summary_metrics.json
```

Add:

```
scenario_results_history.json
trigger_history.json
scenario_catalog_enriched.json
```

That’s it.

Total files = **11 datasets**

Which is still very manageable.

---

# 7. Why This Design Is Powerful

Most evaluation systems only produce:

```
accuracy score
```

Your system will produce:

```
performance trend
failure clusters
risk velocity
trigger evolution
release impact analysis
```

That is exactly what **enterprise AI evaluation systems need.**




This dataset is the **single most important new dataset** for EaaS v3 because it connects:

* scenarios
* orchestrator decisions
* scoring metrics
* historical analysis

Once this exists, everything else (scoring, trends, triggers, reports) works naturally.


---

# scenario_results_history.json

## Purpose

This dataset records the **evaluation result of each scenario during each evaluation run**.

It provides the detailed data required to compute:

* classification accuracy
* resolution path accuracy
* outcome accuracy
* latency metrics
* human review rate
* high-risk failures
* weighted failure rates
* trend analysis across runs

Each record represents **one scenario evaluated during one run**.

---

# Schema

Each row contains the following fields.

## Identification

```json
run_id
scenario_id
target_version
```

| Field          | Description                       |
| -------------- | --------------------------------- |
| run_id         | Evaluation run identifier         |
| scenario_id    | Unique scenario identifier        |
| target_version | Version of orchestrator evaluated |

---

## Expected Behavior (Ground Truth)

These come from the scenario definition and decision rules.

```json
expected_issue_type
expected_resolution_path
expected_outcome
```

| Field                    | Description                  |
| ------------------------ | ---------------------------- |
| expected_issue_type      | Correct issue classification |
| expected_resolution_path | Correct sequence of agents   |
| expected_outcome         | Correct final outcome        |

These values correspond directly to your **decision rules file**.

---

## Orchestrator Output

These fields record what the system actually produced.

```json
predicted_issue_type
predicted_resolution_path
predicted_outcome
```

| Field                     | Description                             |
| ------------------------- | --------------------------------------- |
| predicted_issue_type      | Issue classification produced by system |
| predicted_resolution_path | Agents called by orchestrator           |
| predicted_outcome         | Final orchestrator result               |

---

## Scoring Flags

These fields are used directly by the scoring engine.

```json
issue_classification_correct
resolution_path_correct
outcome_correct
```

| Field                        | Description                                 |
| ---------------------------- | ------------------------------------------- |
| issue_classification_correct | predicted_issue_type == expected_issue_type |
| resolution_path_correct      | predicted path matches expected path        |
| outcome_correct              | predicted_outcome == expected_outcome       |

---

## Operational Metrics

```json
latency_ms
human_review_required
```

| Field                 | Description                 |
| --------------------- | --------------------------- |
| latency_ms            | Total response time         |
| human_review_required | True if escalation required |

---

## Safety Signals

```json
hallucination_flag
policy_violation_flag
```

| Field                 | Description                      |
| --------------------- | -------------------------------- |
| hallucination_flag    | Unsupported conclusion generated |
| policy_violation_flag | Response violates policy         |

These should be extremely rare in deterministic orchestrators.

---

## Scenario Risk Metadata

This allows the evaluator to detect **high-risk failures**.

```json
severity_weight
business_risk
failure_category
```

| Field            | Description                |
| ---------------- | -------------------------- |
| severity_weight  | Numerical importance (1–5) |
| business_risk    | low / medium / high        |
| failure_category | scenario type              |

Example categories:

```
status_check
delivery_delay
lost_package
warehouse_delay
churn_risk
refund_case
```

---

# Example Dataset (MVP)

Below is a small example using your real scenario structure.

```json
[
  {
    "run_id": "RUN_2026_01_24",
    "scenario_id": "SCN_001",
    "target_version": "v1.1",

    "expected_issue_type": "simple_status_check",
    "expected_resolution_path": ["shipping_update_agent"],
    "expected_outcome": "provide_delivery_update",

    "predicted_issue_type": "simple_status_check",
    "predicted_resolution_path": ["shipping_update_agent"],
    "predicted_outcome": "provide_delivery_update",

    "issue_classification_correct": true,
    "resolution_path_correct": true,
    "outcome_correct": true,

    "latency_ms": 610,
    "human_review_required": false,

    "hallucination_flag": false,
    "policy_violation_flag": false,

    "severity_weight": 1,
    "business_risk": "low",
    "failure_category": "status_check"
  },

  {
    "run_id": "RUN_2026_01_24",
    "scenario_id": "SCN_002",
    "target_version": "v1.1",

    "expected_issue_type": "delivery_delay",
    "expected_resolution_path": [
      "shipping_update_agent",
      "apology_message_agent"
    ],
    "expected_outcome": "acknowledge_delay_and_update_eta",

    "predicted_issue_type": "delivery_delay",
    "predicted_resolution_path": [
      "shipping_update_agent",
      "apology_message_agent"
    ],
    "predicted_outcome": "acknowledge_delay_and_update_eta",

    "issue_classification_correct": true,
    "resolution_path_correct": true,
    "outcome_correct": true,

    "latency_ms": 740,
    "human_review_required": false,

    "hallucination_flag": false,
    "policy_violation_flag": false,

    "severity_weight": 2,
    "business_risk": "medium",
    "failure_category": "delivery_delay"
  },

  {
    "run_id": "RUN_2026_01_24",
    "scenario_id": "SCN_003",
    "target_version": "v1.1",

    "expected_issue_type": "lost_package",
    "expected_resolution_path": [
      "refund_agent",
      "apology_message_agent"
    ],
    "expected_outcome": "issue_refund_and_notify_customer",

    "predicted_issue_type": "lost_package",
    "predicted_resolution_path": [
      "refund_agent",
      "apology_message_agent"
    ],
    "predicted_outcome": "issue_refund_and_notify_customer",

    "issue_classification_correct": true,
    "resolution_path_correct": true,
    "outcome_correct": true,

    "latency_ms": 830,
    "human_review_required": false,

    "hallucination_flag": false,
    "policy_violation_flag": false,

    "severity_weight": 5,
    "business_risk": "high",
    "failure_category": "lost_package"
  },

  {
    "run_id": "RUN_2026_01_24",
    "scenario_id": "SCN_004",
    "target_version": "v1.1",

    "expected_issue_type": "delivery_delay_with_churn_risk",
    "expected_resolution_path": [
      "shipping_update_agent",
      "apology_message_agent",
      "escalation_agent"
    ],
    "expected_outcome": "resolve_delay_and_prevent_churn",

    "predicted_issue_type": "delivery_delay",
    "predicted_resolution_path": [
      "shipping_update_agent",
      "apology_message_agent"
    ],
    "predicted_outcome": "acknowledge_delay_and_update_eta",

    "issue_classification_correct": false,
    "resolution_path_correct": false,
    "outcome_correct": false,

    "latency_ms": 845,
    "human_review_required": false,

    "hallucination_flag": false,
    "policy_violation_flag": false,

    "severity_weight": 4,
    "business_risk": "high",
    "failure_category": "churn_risk"
  }
]
```

---

# Why This Dataset Is So Powerful

This single dataset enables:

### Accuracy Metrics

```
classification accuracy
path accuracy
outcome accuracy
```

---

### Risk Analysis

```
high risk failures
weighted failure rate
churn-risk scenario failures
```

---

### Performance Metrics

```
latency
human review rate
```

---

### Trend Analysis

Across runs you can analyze:

```
scenario stability
failure clusters
regression detection
```


# scenario_catalog_enriched.json

## Purpose

This dataset defines the **risk profile and business impact of each evaluation scenario**.

It allows the evaluator to:

• weight failures by severity
• detect high-risk failures
• compute weighted failure rates
• identify business-critical regressions
• prioritize remediation

Without this file, all failures look equal.
With it, the evaluator understands **business impact.**

---

# Why This Dataset Matters

Example:

Failing a **status check**:

```
customer inconvenience
```

Failing a **lost package refund case**:

```
financial loss
customer churn risk
brand damage
```

These are not equivalent.

This file tells the evaluator:

```
which failures matter most
```

---

# Schema

Each scenario has **one entry** in this catalog.

---

## Identification

```json
scenario_id
scenario_name
description
```

| Field         | Description                   |
| ------------- | ----------------------------- |
| scenario_id   | Unique identifier             |
| scenario_name | Human-readable scenario       |
| description   | Short description of scenario |

---

## Scenario Classification

```json
issue_type
failure_category
journey_stage
```

| Field            | Description                     |
| ---------------- | ------------------------------- |
| issue_type       | Issue classification            |
| failure_category | Broad scenario group            |
| journey_stage    | Where failure occurs in journey |

Example journey stages:

```
pre_delivery
delivery
post_delivery
customer_support
```

---

## Risk Signals

```json
severity_weight
business_risk
customer_impact
```

| Field           | Description                                 |
| --------------- | ------------------------------------------- |
| severity_weight | Numerical weight (1–5)                      |
| business_risk   | low / medium / high                         |
| customer_impact | inconvenience / service_failure / financial |

Severity scale:

| Weight | Meaning                |
| ------ | ---------------------- |
| 1      | Minor inconvenience    |
| 2      | Moderate issue         |
| 3      | Service failure        |
| 4      | Customer churn risk    |
| 5      | Financial / brand risk |

---

## Financial Impact

```json
refund_exposure_usd
churn_risk_trigger
brand_risk
```

| Field               | Description                        |
| ------------------- | ---------------------------------- |
| refund_exposure_usd | Potential financial cost           |
| churn_risk_trigger  | Whether scenario impacts retention |
| brand_risk          | reputational risk level            |

---

# Example Dataset (MVP)

This example matches your current scenario design.

```json
[
  {
    "scenario_id": "SCN_001",
    "scenario_name": "Simple Status Check",
    "description": "Customer asks where their order is while it is still in transit.",

    "issue_type": "simple_status_check",
    "failure_category": "status_check",
    "journey_stage": "delivery",

    "severity_weight": 1,
    "business_risk": "low",
    "customer_impact": "inconvenience",

    "refund_exposure_usd": 0,
    "churn_risk_trigger": false,
    "brand_risk": "low"
  },

  {
    "scenario_id": "SCN_002",
    "scenario_name": "Delivery Delay",
    "description": "Package delayed due to shipping carrier issue.",

    "issue_type": "delivery_delay",
    "failure_category": "delivery_delay",
    "journey_stage": "delivery",

    "severity_weight": 2,
    "business_risk": "medium",
    "customer_impact": "service_failure",

    "refund_exposure_usd": 0,
    "churn_risk_trigger": false,
    "brand_risk": "medium"
  },

  {
    "scenario_id": "SCN_003",
    "scenario_name": "Lost Package",
    "description": "Package confirmed lost in transit requiring refund.",

    "issue_type": "lost_package",
    "failure_category": "lost_package",
    "journey_stage": "delivery",

    "severity_weight": 5,
    "business_risk": "high",
    "customer_impact": "financial",

    "refund_exposure_usd": 120,
    "churn_risk_trigger": true,
    "brand_risk": "high"
  },

  {
    "scenario_id": "SCN_004",
    "scenario_name": "Delivery Delay With Churn Risk",
    "description": "High-value customer experiencing delivery delay.",

    "issue_type": "delivery_delay_with_churn_risk",
    "failure_category": "churn_risk",
    "journey_stage": "customer_support",

    "severity_weight": 4,
    "business_risk": "high",
    "customer_impact": "service_failure",

    "refund_exposure_usd": 0,
    "churn_risk_trigger": true,
    "brand_risk": "high"
  },

  {
    "scenario_id": "SCN_005",
    "scenario_name": "Item Not Received",
    "description": "Order marked delivered but customer reports missing item.",

    "issue_type": "item_not_received",
    "failure_category": "post_delivery_issue",
    "journey_stage": "post_delivery",

    "severity_weight": 4,
    "business_risk": "high",
    "customer_impact": "service_failure",

    "refund_exposure_usd": 45,
    "churn_risk_trigger": true,
    "brand_risk": "high"
  }
]
```

---

# How This File Works With the Scoring Engine

When scoring scenarios:

```
scenario_results_history
        +
scenario_catalog_enriched
```

The evaluator can compute:

### Weighted Failure Rate

Example:

```
lost package failure weight = 5
status check failure weight = 1
```

So a lost package failure counts **5× more**.

---

### High Risk Failure Detection

From scoring engine logic:

```
if severity_weight >= 4
→ high risk failure
```

---

### Business Risk Reporting

Your executive report can now say:

```
3 failures detected
2 were high-risk churn scenarios
```

This makes the evaluation system **much more realistic**.

---

# How Large Should This Dataset Be?

Keep it small.

For MVP:

```
10 scenarios
```

That’s perfect.

Example groups:

```
status checks
delivery delays
warehouse delays
lost packages
post delivery issues
churn-risk cases
```

---

# Current EaaS v3 Dataset Architecture

After this step your agent will have:

```
customers.json
orders.json
logistics_api.json
marketing_signals.json

journey_scenarios.json
scenario_catalog_enriched.json

specialist_agents.json
orchestrator_decision_rules.json

evaluation_runs.json
run_summary_metrics.json

scenario_results_history.json
```

This is a **beautifully balanced dataset architecture.**

Not too big.

But extremely powerful.



# trigger_history.json

## Purpose

This dataset records **system trigger events detected during evaluation runs**.

Triggers represent **risk conditions** identified by the scoring engine.

Examples:

* regression detected
* high-risk failures
* latency degradation
* elevated human review rate

The trigger history allows the evaluator to detect:

* emerging risks
* persistent system weaknesses
* resolved issues
* system stability trends

---

# Why This Dataset Is Important

A typical benchmark only reports:

```
accuracy
```

But a real evaluation system must detect:

```
system instability
risk escalation
regression patterns
```

This dataset enables the evaluator to answer questions like:

```
Did a regression occur?
Was it temporary or persistent?
Did a fix resolve the issue?
```

---

# Schema

Each record represents **trigger conditions detected in a run**.

---

## Run Identification

```json
run_id
run_date
target_version
```

| Field          | Description               |
| -------------- | ------------------------- |
| run_id         | Evaluation run identifier |
| run_date       | Date of run               |
| target_version | Version evaluated         |

---

## Trigger Flags

These fields represent **risk signals generated by the scoring engine**.

```json
regression_trigger
critical_risk_trigger
low_evaluation_score_trigger
avg_latency_trigger
p95_latency_trigger
high_human_review_trigger
hallucination_trigger
policy_violation_trigger
weighted_failure_trigger
```

Each field is boolean.

---

## Trigger Summary

These fields provide **quick interpretation of system health**.

```json
trigger_count
persistent_trigger_count
resolved_trigger_count
system_status
```

| Field                    | Description                           |
| ------------------------ | ------------------------------------- |
| trigger_count            | Total triggers this run               |
| persistent_trigger_count | Triggers continuing from previous run |
| resolved_trigger_count   | Triggers resolved since last run      |
| system_status            | healthy / warning / critical          |

---

## Trigger Notes

```json
notes
```

Short explanation for unusual conditions.

---

# Example Dataset

This example aligns with your historical runs.

```json
[
  {
    "run_id": "RUN_2025_12_20",
    "run_date": "2025-12-20",
    "target_version": "v0.9",

    "regression_trigger": false,
    "critical_risk_trigger": true,
    "low_evaluation_score_trigger": true,
    "avg_latency_trigger": false,
    "p95_latency_trigger": false,
    "high_human_review_trigger": true,
    "hallucination_trigger": false,
    "policy_violation_trigger": false,
    "weighted_failure_trigger": true,

    "trigger_count": 4,
    "persistent_trigger_count": 0,
    "resolved_trigger_count": 0,
    "system_status": "critical",

    "notes": "Initial pre-release testing shows multiple system weaknesses"
  },

  {
    "run_id": "RUN_2025_12_27",
    "run_date": "2025-12-27",
    "target_version": "v0.9",

    "regression_trigger": false,
    "critical_risk_trigger": false,
    "low_evaluation_score_trigger": false,
    "avg_latency_trigger": false,
    "p95_latency_trigger": false,
    "high_human_review_trigger": false,
    "hallucination_trigger": false,
    "policy_violation_trigger": false,
    "weighted_failure_trigger": false,

    "trigger_count": 0,
    "persistent_trigger_count": 0,
    "resolved_trigger_count": 4,
    "system_status": "warning",

    "notes": "Major stability improvements after rule tuning"
  },

  {
    "run_id": "RUN_2026_01_03",
    "run_date": "2026-01-03",
    "target_version": "v1.0",

    "regression_trigger": false,
    "critical_risk_trigger": false,
    "low_evaluation_score_trigger": false,
    "avg_latency_trigger": false,
    "p95_latency_trigger": false,
    "high_human_review_trigger": false,
    "hallucination_trigger": false,
    "policy_violation_trigger": false,
    "weighted_failure_trigger": false,

    "trigger_count": 0,
    "persistent_trigger_count": 0,
    "resolved_trigger_count": 0,
    "system_status": "healthy",

    "notes": "Production readiness confirmed"
  },

  {
    "run_id": "RUN_2026_01_17",
    "run_date": "2026-01-17",
    "target_version": "v1.1",

    "regression_trigger": true,
    "critical_risk_trigger": true,
    "low_evaluation_score_trigger": false,
    "avg_latency_trigger": false,
    "p95_latency_trigger": true,
    "high_human_review_trigger": true,
    "hallucination_trigger": false,
    "policy_violation_trigger": false,
    "weighted_failure_trigger": true,

    "trigger_count": 4,
    "persistent_trigger_count": 0,
    "resolved_trigger_count": 0,
    "system_status": "critical",

    "notes": "Regression detected after churn-risk threshold adjustments"
  },

  {
    "run_id": "RUN_2026_01_24",
    "run_date": "2026-01-24",
    "target_version": "v1.1",

    "regression_trigger": false,
    "critical_risk_trigger": false,
    "low_evaluation_score_trigger": false,
    "avg_latency_trigger": false,
    "p95_latency_trigger": false,
    "high_human_review_trigger": false,
    "hallucination_trigger": false,
    "policy_violation_trigger": false,
    "weighted_failure_trigger": false,

    "trigger_count": 0,
    "persistent_trigger_count": 0,
    "resolved_trigger_count": 4,
    "system_status": "healthy",

    "notes": "Regression resolved after rule correction"
  }
]
```

---

# How This Works With the Scoring Engine

The scoring engine generates triggers like:

```
critical_risk_trigger
high_human_review_trigger
latency_trigger
regression_trigger
```

Those triggers are stored in this dataset.

Over time the evaluator can detect:

### Persistent Risks

```
trigger appears in multiple runs
```

### Resolved Risks

```
trigger disappears after fix
```

### Emerging Instability

```
trigger frequency increasing
```

---

# Why This Makes EaaS v3 Special

Most AI evaluation systems only report:

```
accuracy
```

EaaS v3 now reports:

```
accuracy
risk exposure
trend stability
trigger evolution
system health
```

This is much closer to **how production reliability engineering works**.

---

# Final v3 Dataset Architecture

Your EaaS v3 now has a **complete dataset system**:

```
customers.json
orders.json
logistics_api.json
marketing_signals.json

journey_scenarios.json
scenario_catalog_enriched.json

specialist_agents.json
orchestrator_decision_rules.json

evaluation_runs.json
run_summary_metrics.json

scenario_results_history.json
trigger_history.json
```

This is a **perfect MVP evaluation environment.**

